In [3]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "acc_mappings" / "acc_mappings_hex"


In [4]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 22 firm-variable mappings across 21 firms.


firm_id,variable,notes,final_choice,num_candidates
ASPO.HE,STD,"Most likely the correct current debt line in the current liabilities section, with leases separately disclosed. Manual review because the same row label also appears in the non-current liabilities section, so the label is duplicated on the sheet.",Balance Sheet::Loans and overdraft facilities,4
BITTI.HE,STD,"Manual review because the apparent total current debt line, 'Interest-bearing loans and borrowings (current)', seems to include lease liabilities in some years. Chose the cleaner non-lease component 'Current loans from financial institutions', but coverage across years may be incomplete.",Balance Sheet::Current loans from financial institutions,5
CONSTI.HE,STD,"STD is ambiguous because current interest-bearing liabilities are split and the sheet has duplicate 'Loans from financial institutions' labels for non-current and current sections. I excluded lease rows per instruction. Chosen rows capture explicit current debt and short-term paper, but manual review is warranted to confirm whether current bank loans/hire purchase debt are already embedded in 'Long Term Debt Maturing within 1 Year' or should be added separately.",Balance Sheet::Long Term Debt Maturing within 1 Year; Balance Sheet::Commercial papers,5
CTY1S.HE,STD,"Selected current-section financing rows as STD. Manual review because 'Loans' is duplicated elsewhere and it is somewhat unclear whether 'Commercial Papers' are already included in current 'Loans', creating possible double-count risk.",Balance Sheet::Loans; Balance Sheet::Commercial Papers,6
DIGIA.HE,STD,"Best semantic match for current debt excluding leases, but confidence is limited because preview values are blank and nearby 'Other financial liabilities' could also capture current borrowings. Excluded lease-related rows per rule.",Balance Sheet::Current interest-bearing liabilities,3
ESENSE.HE,STD,"Selected the explicit non-lease current maturities debt line. Manual review because current-liabilities section also contains duplicated 'Borrowings' rows, creating ambiguity about whether a broader current debt total exists and should replace this line.",Balance Sheet::Debt - Long-Term - Maturities - within 1 Year,2
FORTUM.HE,TXP,"Chose a balance-sheet payable stock outside the shortlist because TXP is defined as taxes payable/current tax liabilities payable, and this is the explicit matching line. | Auto-flag: best candidate confidence 0.08 < 0.65",Balance Sheet::Income Taxes - Payable - Short-Term,1
HIAB.HE,STD,"Selected non-lease current debt components and excluded 'S/T Finance Leases' per rule. Manual review needed because 'Other Interest-Bearing Liabilities' may also represent current debt in some years, but including it risks double counting against the current-portion breakdown.",Balance Sheet::Current Portion of Interest-Bearing Liabilities - Balancing value; Balance Sheet::Schuldschein Loans,7
ILKKA2.HE,TXP,"Selected 'Income Tax' as the likely current tax payable line, but the same label also appears in the assets section, so the mapping is ambiguous without row position context.",Balance Sheet::Income Tax,3
KREATE.HE,STD,"Selected the cleanest current debt total and excluded explicit lease rows. Manual review because the same exact row label appears twice in the sheet preview, creating ambiguity about whether the extracted row is unique and whether any short-term debt components are separately reported.",Balance Sheet::Short-Term Debt Interest-Bearing Debt,3
